# Stage 05 - RFdiffusion binder backbones (Kaggle)

Kaggle port of `05_rfdiffusion_binder.ipynb`. Same target, same contigs, same
hotspots, same noise scale: only the transport and the Python environment differ.

**Why this notebook exists.** Colab could not provision a runtime on 9 Nov 2026
with the pilot still unrun. Kaggle has GPU quota, but its image is built from the
Colab runtime image and is **Python 3.13**, where `pip install dgl` silently
resolves to dgl 0.1.3 (a 2018 universal wheel) whose `utils.py` does
`from collections import Mapping` - removed in Python 3.10. RFdiffusion then
fails at import. Python 3.12 resolves to dgl 2.1.0, so cell 2 builds a Python
3.12 environment and every later cell uses it.

**Manual steps** (also in `docs/colab_runbook.md`):

1. Session options: **Accelerator = GPU T4**, **Internet = On** (both need phone
   verification).
2. **+ Add Input -> Datasets -> New Dataset**, upload
   `data/targets_relaxed/7ow3_L_G6D_chi1m70_chi2p120_relaxed_crop.pdb`.
   The slug does not matter: cell 3 searches `/kaggle/input` for the filename
   and verifies its SHA-256 against the constant in cell 1.
3. Run cells 1-6 (cell 6 is the pilot). Stop there and ingest on the Mac.
4. Cell 6 is production to the cutoff; cell 7 zips the output for download.\n5. On the Mac, relabel and score in one step:\n   `make backbones DRIVE_DIR=<unzipped>/stage05 RELABEL=1`

Upstream: sokrypton/ColabDesign `rf/examples/diffusion.ipynb` (setup logic),
RFdiffusion from sokrypton's fork.

In [ ]:
#@title 1. Parameters - every knob lives here

# --- target geometry (see docs/design_plan.md) ---
TARGET_NAME = "7ow3_L_G6D_chi1m70_chi2p120_relaxed_crop.pdb"
# Written by scripts/05a_stage_to_drive.py into TARGETS.tsv. The file is uploaded
# to Kaggle by hand, so this constant is what ties these backbones to stage 04.
TARGET_SHA256 = "1d3aa77d4be830aa211868ec8775333d3e9e1adf1e75bb9f5a27aab6f3015e9b"

TARGET_CONTIGS = ["A1-180/0", "C1-10/0"]  # HLA alpha1/alpha2, then peptide
BINDER_LENGTH = "60-75"                   # sampled per design

# Hotspots centred on the mutation. C5/C6/C7 = Ala/Asp/Gly, so Asp6 is the only
# side chain of substance here; 3 is within the 3-6 the RFdiffusion authors
# recommend, and the model expects to make more contacts than specified.
HOTSPOTS = "C5,C6,C7"

# --- RFdiffusion sampling ---
# 0.5 per the RFdiffusion README for protein-protein interfaces: "we've
# consistently seen that reducing the noise added at inference improves
# designs... 0.5 is probably a good compromise". Costs topological diversity.
NOISE_SCALE = 0.5
DIFFUSER_T = 50
CHECKPOINT = "Complex_base_ckpt.pt"
USE_SEED = True

# --- batching ---
# Deliberately small. Designs are only copied out of RFdiffusion's output
# directory and logged once a run_inference.py call returns, so the batch size
# is the amount of work a dropped session can destroy. Kaggle was reporting
# connection problems on 9 Nov 2026, so 2 (something saved every ~5 min) beats
# 8 (~20 min at risk). The cost is reloading the model once per call.
BATCH_SIZE = 2
PILOT_SEED = 0

# --- production cutoff ---
# Kaggle runs in UTC. Set by the user: 20:00 UTC Mon 9 Nov = 15:00 EST.
# Deliberately earlier than the deadline allows - a submitted set of designs is
# worth more than a larger set that stages 06-11 run out of time to package.
CUTOFF_UTC = "2026-11-09T20:00:00"

# --- Kaggle paths ---
# /kaggle/working is the only persisted writable directory and is saved as the
# notebook's output. The virtualenv goes to /tmp deliberately: it is ~3 GB and
# has no business in the saved output.
BASE = "/kaggle/working"
VENV = "/tmp/venv312"

print(f"contigs : [{' '.join(TARGET_CONTIGS + [BINDER_LENGTH])}]")
print(f"hotspots: [{HOTSPOTS}]")
print(f"noise   : {NOISE_SCALE}  T={DIFFUSER_T}  ckpt={CHECKPOINT}")
print(f"cutoff  : {CUTOFF_UTC} UTC")

In [ ]:
#@title 2. Python 3.12 environment + RFdiffusion (~8 min)
import os
import subprocess
import sys
import time
from pathlib import Path

os.chdir(BASE)


def sh(command, check=True):
    """Run a shell command, streaming nothing but reporting failure loudly."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"failed ({result.returncode}): {command}")
    return result


# --- the interpreter question -------------------------------------------
# dgl publishes no wheel for 3.13, and pip falls back to dgl 0.1.3 (2018),
# which cannot be imported at all. 3.12 resolves to dgl 2.1.0. So: use the
# system interpreter if it is new enough to run RFdiffusion and old enough to
# have dgl, otherwise build a 3.12 venv with uv.
if sys.version_info[:2] <= (3, 12):
    PY = sys.executable
    print(f"system python {sys.version.split()[0]} is usable; no venv needed")
else:
    print(f"system python is {sys.version.split()[0]}; building a 3.12 venv")
    sh(f"{sys.executable} -m pip install -q uv")
    sh("uv python install 3.12")
    sh(f"uv venv --python 3.12 {VENV}")
    PY = f"{VENV}/bin/python"

print(sh(f"{PY} -V").stdout.strip())

UV = f"uv pip install --python {PY}"

# --- RFdiffusion source and checkpoints ---------------------------------
# Only the two checkpoints the plan uses are fetched: Complex_base is the
# default and Complex_beta is the documented remedy if the pilot gate fails.
if not Path("RFdiffusion").is_dir():
    print("cloning RFdiffusion...")
    sh("git clone -q https://github.com/sokrypton/RFdiffusion.git")

Path("RFdiffusion/models").mkdir(parents=True, exist_ok=True)
CKPTS = {
    "Complex_base_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt",
    "Complex_beta_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt",
}
for name, url in CKPTS.items():
    destination = Path("RFdiffusion/models") / name
    if not destination.is_file():
        print(f"downloading {name}...")
        sh(f"wget -q -O {destination} {url}")

if not Path("schedules").is_dir():
    print("downloading beta schedules...")
    sh("wget -q -O schedules.zip https://files.ipd.uw.edu/krypton/schedules.zip")
    sh("unzip -q -o schedules.zip && rm schedules.zip")

# --- dependencies -------------------------------------------------------
# torch is pinned to 2.4.1+cu124 because that is the build dgl's CUDA wheels
# are compiled against; a mismatch here is the usual cause of a segfault
# inside the SE3 transformer rather than an honest import error.
if not Path(f"{BASE}/.deps_done").exists():
    print("installing torch (large; a few minutes)...")
    sh(f"{UV} torch==2.4.1 --index-url https://download.pytorch.org/whl/cu124")
    # Two sources, in order of preference. data.dgl.ai carries the CUDA builds;
    # PyPI's dgl is a CPU build, which imports fine but cannot move a graph to
    # the GPU - so it is a fallback, and the probe below is what catches it.
    # The CUDA index was unreachable from the dev machine, hence the fallback.
    print("installing dgl...")
    dgl_cuda = sh(f"{UV} --no-deps dgl -f "
                  "https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html", check=False)
    if dgl_cuda.returncode != 0:
        print("  CUDA index failed; falling back to PyPI dgl==2.1.0")
        print(dgl_cuda.stderr[-800:])
        sh(f"{UV} --no-deps dgl==2.1.0")
    print("installing the rest...")
    sh(f"{UV} hydra-core omegaconf icecream pyrsistent decorator pynvml "
       f"scipy networkx psutil requests tqdm pandas opt_einsum_fx e3nn==0.5.5")
    sh(f"{UV} git+https://github.com/NVIDIA/dllogger")
    print("installing SE3Transformer...")
    sh(f"cd RFdiffusion/env/SE3Transformer && {UV} .")
    Path(f"{BASE}/.deps_done").touch()
else:
    print("dependencies already installed")

# The import that has failed every previous attempt. Checked here, before any
# GPU time is spent, so a broken environment costs seconds rather than a batch.
# Importing dgl is necessary but not sufficient: the SE3 transformer moves
# graphs onto the GPU, and a CPU-only dgl build fails there rather than at
# import. So the probe builds a graph and moves it.
PROBE = (
    "import dgl, torch;"
    "g = dgl.graph((torch.tensor([0,1]), torch.tensor([1,0])));"
    "g = g.to('cuda');"
    "print('dgl', dgl.__version__, 'torch', torch.__version__,"
    "      'cuda', torch.cuda.is_available(), 'graph_on', g.device)"
)
probe = sh(f'cd {BASE}/RFdiffusion && DGLBACKEND=pytorch {PY} -c "{PROBE}"', check=False)
print(probe.stdout.strip() or probe.stderr[-1500:])
if probe.returncode != 0:
    raise RuntimeError(
        "dgl cannot put a graph on the GPU - stop here, do not run the pilot"
    )

print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv").stdout.strip())

In [ ]:
#@title 3. Locate and verify the target
import hashlib
import shutil

matches = sorted(Path("/kaggle/input").rglob(TARGET_NAME))
if not matches:
    raise FileNotFoundError(
        f"{TARGET_NAME} not found under /kaggle/input - add it with "
        "+ Add Input -> Datasets -> New Dataset"
    )

source = matches[0]
digest = hashlib.sha256(source.read_bytes()).hexdigest()
if digest != TARGET_SHA256:
    raise ValueError(
        f"checksum mismatch for {source}\n  got      {digest}\n  expected {TARGET_SHA256}"
    )

TARGET_LOCAL = f"{BASE}/target.pdb"
shutil.copy2(source, TARGET_LOCAL)

BACKBONE_DIR = Path(BASE) / "backbones"
BACKBONE_DIR.mkdir(exist_ok=True)
RUN_LOG = Path(BASE) / "run_log.tsv"

print(f"target  : {source}")
print(f"sha256  : {digest[:16]}... verified against TARGET_SHA256")
print(f"outputs : {BACKBONE_DIR}")

In [ ]:
#@title 5. Generation driver
import csv
from datetime import datetime, timezone

CONTIGS = TARGET_CONTIGS + [BINDER_LENGTH]
CONTIG_STR = ' '.join(CONTIGS)
LOG_FIELDS = ('batch', 'seed', 'design', 'filename', 'contigs', 'hotspots',
              'noise_scale', 'diffuser_T', 'checkpoint', 'elapsed_s', 'written_utc')


def _log_rows():
    if not RUN_LOG.exists():
        return []
    with open(RUN_LOG) as handle:
        return list(csv.DictReader(handle, delimiter='\t'))


def _append_log(rows):
    exists = RUN_LOG.exists()
    with open(RUN_LOG, 'a', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=LOG_FIELDS, delimiter='\t')
        if not exists:
            writer.writeheader()
        writer.writerows(rows)


def next_batch():
    """Resume rather than restart: a restarted session picks up here."""
    rows = _log_rows()
    return (max(int(r['batch']) for r in rows) + 1) if rows else 0


def harvest():
    """Recover designs RFdiffusion wrote but no batch got to log.

    run_inference.py writes each design as it finishes, so a batch killed
    mid-flight still leaves finished structures on disk. Without this they
    would be silently discarded, which on an unreliable session is most of what
    there is to lose. Seeds are unknown for recovered designs and recorded as
    'orphan'.
    """
    known = {row['filename'] for row in _log_rows()}
    rows, found = [], 0
    for pdb in sorted(Path(f'{BASE}/outputs').glob('batch*_*.pdb')):
        # Keep the name RFdiffusion's prefix gave it. Renaming would let an
        # already-logged design be harvested a second time under a new name.
        name = pdb.name
        if name in known or (BACKBONE_DIR / name).exists():
            continue
        shutil.copy2(pdb, BACKBONE_DIR / name)
        # generate() passes seed=batch whenever seed is None, and the pilot uses
        # seed 0 in batch 0, so the seed is recoverable from the batch number
        # for every call this notebook makes. Recorded, not guessed at.
        batch_number = int(name.removeprefix('batch').split('_')[0])
        rows.append({
            'batch': batch_number, 'seed': batch_number, 'design': found,
            'filename': name,
            'contigs': CONTIG_STR, 'hotspots': HOTSPOTS,
            'noise_scale': NOISE_SCALE, 'diffuser_T': DIFFUSER_T,
            'checkpoint': CHECKPOINT, 'elapsed_s': '',
            'written_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        })
        found += 1
    if rows:
        _append_log(rows)
    print(f'harvested {found} orphaned design(s)')
    return found


def generate(n=BATCH_SIZE, seed=None, checkpoint=CHECKPOINT, hotspots=HOTSPOTS):
    """Run one RFdiffusion batch, writing backbones and log rows to disk.

    Every flag is spelled out rather than assembled by a helper, so the printed
    command is a complete record of what produced these coordinates.
    """
    batch = next_batch()
    seed = batch if seed is None else seed
    prefix = f'{BASE}/outputs/batch{batch}'
    Path(f'{BASE}/outputs').mkdir(exist_ok=True)

    flags = [
        f'inference.output_prefix={prefix}',
        f'inference.input_pdb={TARGET_LOCAL}',
        f'inference.num_designs={n}',
        f'inference.ckpt_override_path={BASE}/RFdiffusion/models/{checkpoint}',
        f'diffuser.T={DIFFUSER_T}',
        f'denoiser.noise_scale_ca={NOISE_SCALE}',
        f'denoiser.noise_scale_frame={NOISE_SCALE}',
        f"'contigmap.contigs=[{CONTIG_STR}]'",
        f"'ppi.hotspot_res=[{hotspots}]'",
    ]
    if USE_SEED:
        flags.append(f'inference.seed={seed}')

    cmd = (f'cd {BASE}/RFdiffusion && DGLBACKEND=pytorch '
           f'{PY} run_inference.py ' + ' '.join(flags))
    print(f'--- batch {batch} (seed {seed}, {n} designs) ---')
    print(cmd)

    started = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - started

    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        # Keep anything it finished before dying, then report the failure.
        harvest()
        raise RuntimeError(f'batch {batch} failed with code {result.returncode}')

    rows, written = [], []
    for i in range(n):
        pdb = Path(f'{prefix}_{i}.pdb')
        if not pdb.exists():
            print(f'  missing {pdb.name}; stopping this batch')
            break
        name = f'batch{batch}_{i}.pdb'
        # Copied verbatim. RFdiffusion returns one continuously renumbered
        # chain; it is relabelled on the Mac by 05_ingest_backbones.py
        # --relabel-contigs, which uses the tested kras_pmhc.nim.relabel_chains
        # rather than a second implementation living in a notebook.
        shutil.copy2(pdb, BACKBONE_DIR / name)
        written.append(name)
        rows.append({
            'batch': batch, 'seed': seed, 'design': i, 'filename': name,
            'contigs': CONTIG_STR, 'hotspots': hotspots,
            'noise_scale': NOISE_SCALE, 'diffuser_T': DIFFUSER_T,
            'checkpoint': checkpoint,
            'elapsed_s': round(elapsed / max(n, 1), 1),
            'written_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        })

    _append_log(rows)
    print(f'  wrote {len(written)} backbones in {elapsed / 60:.1f} min '
          f'({elapsed / max(len(written), 1):.0f} s each)')
    return written


print(f'contigs: [{CONTIG_STR}]')
print(f'resuming at batch {next_batch()}')

In [ ]:
#@title 6. Pilot - 8 designs, then STOP and ingest on the Mac
written = generate(n=BATCH_SIZE, seed=PILOT_SEED)

# Residue counts only - 190 target + 60-75 binder is what a correct output
# looks like. Everything that decides anything happens on the Mac, where
# 05_ingest_backbones.py applies the Levy (2010) rSASA classification; there
# must not be a second implementation of it here.
for name in written:
    text = (BACKBONE_DIR / name).read_text()
    residues = {line[21:27] for line in text.splitlines() if line.startswith('ATOM')}
    print(f'  {name}: {len(residues)} residues (expect 250-265)')

In [ ]:
#@title 7. Production - batches of 8 until the cutoff
cutoff = datetime.fromisoformat(CUTOFF_UTC).replace(tzinfo=timezone.utc)
print(f'generating until {cutoff.isoformat()} UTC')

while True:
    now = datetime.now(timezone.utc)
    remaining = (cutoff - now).total_seconds()
    if remaining <= 0:
        print('cutoff reached')
        break
    print(f'\n[{now.strftime("%H:%M:%S")}Z] {remaining / 60:.0f} min left')
    try:
        generate(n=BATCH_SIZE)
    except RuntimeError as error:
        # One failed batch should not end the session: the log is already on
        # disk and the next batch gets a fresh seed.
        print(f'batch failed, continuing: {error}')

print(f'\ntotal backbones: {len(_log_rows())}')

In [ ]:
#@title 8. Package for download
# Kaggle has no synced folder, so the results come back as one archive. Keeping
# the backbones/ + run_log.tsv layout means scripts/05_ingest_backbones.py reads
# the unzipped directory with no change:
#     make backbones DRIVE_DIR=<unzipped path>
# The archive must carry run_log.tsv as well as the backbones: the ingest
# script reads each backbone's seed and batch from the log and refuses to run
# without it. Staged into one directory so the zip unpacks ready to ingest.
# Run this as often as you like - after the pilot and every 30-45 min during
# production - and download the zip. Files in /kaggle/working do not reliably
# survive a session ending, so a downloaded archive is the only real insurance.
# RFdiffusion's raw output directory is deliberately NOT deleted: harvest()
# needs it to recover designs from a batch that died.
harvest()

stage = Path(BASE) / 'stage05'
shutil.rmtree(stage, ignore_errors=True)
stage.mkdir()
shutil.copytree(BACKBONE_DIR, stage / 'backbones')
shutil.copy2(RUN_LOG, stage / 'run_log.tsv')

archive = shutil.make_archive(f'{BASE}/kras_pmhc_stage05', 'zip',
                              root_dir=BASE, base_dir='stage05')
n = len(list((stage / 'backbones').glob('*.pdb')))
print(f'wrote {archive} ({n} backbones + run_log.tsv)')
print('Download it from the Output tab, unzip, then on the Mac:')
print('  make backbones DRIVE_DIR=<unzipped>/stage05')